# Vector Search

**Definition:** A vector database (or vector *index*, for our small-scale purposes) stores each chunk's embedding alongside the chunk itself, and answers a search by comparing a query's embedding against every stored one — returning the chunks whose meaning is closest to the query's meaning.

Here we build a minimal, dependency-free `VectorIndex`: two parallel lists (`vectors[i]` is the embedding of `documents[i]`), and a brute-force scan that scores the query against every stored vector. That's `O(n)` per search — fine for a handful of chunks. A production vector database (Pinecone, pgvector, Chroma, ...) swaps the brute-force scan for an approximate-nearest-neighbour index so it stays fast at millions of vectors, but the interface — add documents, search, get the closest matches back — is the same.


In [1]:
%pip install -q voyageai python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Client setup (from 002_embeddings)
from dotenv import load_dotenv
import voyageai

load_dotenv()

client = voyageai.Client()


def chunk_by_section(document_text):
    import re

    pattern = r"\n## "
    return re.split(pattern, document_text)


def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    is_list = isinstance(chunks, list)
    input = chunks if is_list else [chunks]
    result = client.embed(input, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]


In [3]:
# VectorIndex implementation
#
# A minimal, dependency-free stand-in for a real vector database. It keeps two
# parallel lists - self.vectors[i] is the embedding of self.documents[i] - and
# answers a search by scoring the query against every stored vector.
import math
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(self, distance_metric: str = "cosine", embedding_fn=None):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError("Embedding function not provided during initialization.")
        vector = self._embedding_fn(document["content"])
        self.add_vector(vector=vector, document=document)

    def add_documents(self, documents: List[Dict[str, Any]], vectors: Optional[List[List[float]]] = None):
        # Bulk version of add_document, to avoid one API call per chunk.
        if vectors is None:
            if not self._embedding_fn:
                raise ValueError("Provide either precomputed vectors or an embedding function.")
            vectors = self._embedding_fn([doc["content"] for doc in documents])

        for vector, document in zip(vectors, documents):
            self.add_vector(vector=vector, document=document)

    def add_vector(self, vector, document: Dict[str, Any]):
        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}")

        self.vectors.append(list(vector))
        self.documents.append(document)

    def search(self, query: Any, k: int = 1) -> List[Tuple[Dict[str, Any], float]]:
        # Accepts raw text (embedded on the fly) or a ready-made vector.
        # Returns (document, distance) pairs — lower distance is a better match.
        if not self.vectors:
            return []

        if isinstance(query, str):
            query_vector = self._embedding_fn(query)
        else:
            query_vector = query

        dist_func = self._cosine_distance if self._distance_metric == "cosine" else self._euclidean_distance

        distances = [(dist_func(query_vector, v), doc) for v, doc in zip(self.vectors, self.documents)]
        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def _euclidean_distance(self, vec1, vec2) -> float:
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _cosine_distance(self, vec1, vec2) -> float:
        # 1 - cosine similarity: 0 (identical direction) to 2 (opposite).
        # Compares direction only, so a long chunk and a short chunk about the
        # same topic still score close.
        mag1 = math.sqrt(sum(x * x for x in vec1))
        mag2 = math.sqrt(sum(x * x for x in vec2))
        if mag1 == 0 or mag2 == 0:
            return 1.0
        dot = sum(p * q for p, q in zip(vec1, vec2))
        similarity = max(-1.0, min(1.0, dot / (mag1 * mag2)))
        return 1.0 - similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}')"


## Building and searching the index

Load `report.md`, chunk it by section, embed every chunk, and store the vectors. Then embed a *question* (`input_type="query"`) and search for the closest chunks.

Notice the query never says "Section 9" or names the heading — matching happens on meaning, so wording that shares no keywords with the chunk can still be the nearest vector.


In [4]:
with open("./report.md", "r") as f:
    text = f.read()

chunks = chunk_by_section(text)
print(f"{len(chunks)} chunks")


15 chunks


In [5]:
embeddings = generate_embedding(chunks, input_type="document")

store = VectorIndex()
store.add_documents([{"content": chunk} for chunk in chunks], vectors=embeddings)

store


VectorIndex(count=15, dim=1024, metric='cosine')

In [6]:
query = "What did the Phase IIa trial of the new compound show?"
query_embedding = generate_embedding(query, input_type="query")

results = store.search(query_embedding, k=2)

for rank, (doc, distance) in enumerate(results, start=1):
    preview = doc["content"][:300].replace("\n", " ")
    print(f"--- Rank {rank} | distance {distance:.4f} ---")
    print(preview + "...\n")


--- Rank 1 | distance 0.5221 ---
Section 9: Pharmaceutical Development - Compound CTX-204b Phase IIa Update  Promising results emerged from the Phase IIa clinical trial (`Trial ID: CTX204b-P2A-001`) for Compound CTX-204b, our lead candidate targeting Receptor Pathway Gamma-7. Interim analysis of data from the initial patient cohort...

--- Rank 2 | distance 0.6734 ---
Section 1: Medical Research - Understanding XDR-471 Syndrome  This year saw significant strides in our understanding of XDR-471 syndrome, a rare neurodegenerative condition previously hampered by diagnostic ambiguity. The team focused on correlating clinical presentations with specific genetic marke...

